# PySpark Capabilities — 1-Hour Class Notebook

**Hands-on PySpark tutorial** using real e-commerce data from **ShopStream** (orders, clickstream, product reviews).

| | |
|---|---|
| **Duration** | 60 minutes |
| **Audience** | Data engineering students who know Python + basic Pandas |
| **Run from** | `hadoop-local-docker/spark/` |
| **Data** | `../data/ecommerce/` — orders, clickstream, reviews |
| **Spark UI** | http://localhost:4040 (after starting a session) |

---

## Class agenda

| Time | Module | PySpark capability |
|------|--------|--------------------|
| 0–5 min | Setup | `SparkSession`, local mode |
| 5–10 min | Pandas vs Spark | When to use each |
| 10–15 min | Read & schema | CSV, types, `printSchema` |
| 15–22 min | Transformations | `select`, `filter`, `withColumn` |
| 22–30 min | Aggregations | `groupBy`, `agg`, pivot |
| 30–38 min | Joins | inner, left, broadcast hint |
| 38–46 min | Window functions | rank, running totals |
| 46–52 min | Spark SQL | register temp views, SQL queries |
| 52–56 min | Performance | lazy eval, cache, `explain` |
| 56–60 min | Pipeline wrap-up | write output, summary |

> **How to use:** Run cells **in order**. Each section shows **PySpark** first, then a **Pandas equivalent** on the same small dataset so you see the API difference.

---
## Module 1 — Setup (0–5 min)

PySpark runs on the **JVM** (Java). Your Python code talks to Spark through the **Driver**; work happens on **Executors** (local threads in `local[*]` mode).

```
  Python notebook (Driver)
        │
        ▼ builds lazy DAG
  Spark executors (local JVM threads)
        │
        ▼
  CSV files on disk
```

In [ ]:
import sys
from pathlib import Path

spark_dir = Path("spark" if Path("spark/spark_helpers.py").exists() else ".").resolve()
if str(spark_dir) not in sys.path:
    sys.path.insert(0, str(spark_dir))

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from spark_helpers import create_spark, data_path, stop_spark

spark = create_spark("ShopStream-PySpark-Class")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Spark UI: http://localhost:4040")

---
## Module 2 — Pandas vs PySpark (5–10 min)

| | **Pandas** | **PySpark** |
|---|-----------|-------------|
| Data model | Single-machine `DataFrame` in RAM | Distributed `DataFrame` (lazy) |
| Execution | Eager — each line runs immediately | Lazy — builds a plan until an **action** |
| Sweet spot | < few GB, exploration on laptop | GB → TB, clusters (EMR, Databricks) |
| API feel | Very similar column names & methods | Similar, but distributed semantics |

**Rule of thumb:** Prototype with Pandas on a sample; productionize with PySpark on the full dataset.

In [ ]:
import pandas as pd

orders_pd = pd.read_csv(data_path("orders.csv"))
print("Pandas shape:", orders_pd.shape)
orders_pd.head(3)

In [ ]:
orders = spark.read.option("header", True).csv(data_path("orders.csv"))
print("PySpark partitions:", orders.rdd.getNumPartitions())
print("Row count (action — triggers Spark job):", orders.count())
orders.show(3, truncate=False)

**Key difference:** `orders.head(3)` in Pandas returns instantly. In PySpark, `.show(3)` is an **action** that runs a distributed job. `.count()` is another action.

---
## Module 3 — Read data & schema (10–15 min)

Real pipelines define **explicit schemas** so bad rows fail fast instead of silently becoming `null`.

In [ ]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("order_date", StringType(), False),
    StructField("status", StringType(), False),
    StructField("city", StringType(), False),
])

orders = spark.read.schema(orders_schema).option("header", True).csv(data_path("orders.csv"))
clickstream = spark.read.option("header", True).csv(data_path("clickstream.csv"))
reviews = spark.read.text(data_path("product_reviews.txt"))

print("=== Orders schema ===")
orders.printSchema()
print("\n=== Clickstream sample ===")
clickstream.show(3, truncate=False)

In [ ]:
# Pandas: dtypes are inferred automatically — fine for small CSVs
print(orders_pd.dtypes)

# PySpark: explicit schema avoids string columns for numeric fields
orders.dtypes

---
## Module 4 — Transformations (15–22 min)

**Transformations are lazy** — they add steps to the DAG but do not run until an action.

Business question: *Compute line revenue and flag high-value orders (> ₹100).*

In [ ]:
enriched = (
    orders
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("is_high_value", F.col("revenue") > 100)
    .filter(F.col("status") != "Cancelled")
    .select("order_id", "customer_id", "product_name", "category", "revenue", "is_high_value", "city")
)

print("PySpark — enriched orders")
enriched.orderBy(F.desc("revenue")).show(truncate=False)

In [ ]:
# Pandas equivalent
enriched_pd = orders_pd.copy()
enriched_pd["revenue"] = enriched_pd["quantity"] * enriched_pd["unit_price"]
enriched_pd["is_high_value"] = enriched_pd["revenue"] > 100
enriched_pd = enriched_pd[enriched_pd["status"] != "Cancelled"][
    ["order_id", "customer_id", "product_name", "category", "revenue", "is_high_value", "city"]
]
enriched_pd.sort_values("revenue", ascending=False).head(8)

| Pandas | PySpark |
|--------|--------|
| `df["col"] = ...` | `.withColumn("col", ...)` |
| `df[df["status"] != "Cancelled"]` | `.filter(F.col("status") != "Cancelled")` |
| `df[["a", "b"]]` | `.select("a", "b")` |

PySpark chains read cleaner for multi-step pipelines and pushes work to the cluster.

---
## Module 5 — Aggregations & pivot (22–30 min)

Business questions:
1. Total revenue and order count **by category**
2. Click actions **pivoted** into columns per customer

In [ ]:
revenue_by_category = (
    enriched
    .groupBy("category")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.count("*").alias("order_lines"),
        F.round(F.avg("revenue"), 2).alias("avg_order_value"),
    )
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by category — PySpark")
revenue_by_category.show(truncate=False)

In [ ]:
# Pandas equivalent
revenue_by_category_pd = (
    enriched_pd.groupby("category")["revenue"]
    .agg(total_revenue="sum", order_lines="count", avg_order_value="mean")
    .round(2)
    .sort_values("total_revenue", ascending=False)
)
revenue_by_category_pd

In [ ]:
# Pivot: one row per customer, columns for each clickstream action
actions_by_customer = (
    clickstream
    .groupBy("customer_id", "action")
    .count()
    .groupBy("customer_id")
    .pivot("action")
    .sum("count")
    .na.fill(0)
)

print("Clickstream pivot — PySpark")
actions_by_customer.orderBy("customer_id").show(truncate=False)

In [ ]:
# Pandas equivalent — pivot_table
clickstream_pd = pd.read_csv(data_path("clickstream.csv"))
actions_pd = (
    clickstream_pd.groupby(["customer_id", "action"]).size()
    .unstack(fill_value=0)
    .reset_index()
)
actions_pd

**At scale:** Pandas pivot loads everything into memory. PySpark pivot triggers a **shuffle** (data moves across partitions) but scales to millions of customers.

---
## Module 6 — Joins (30–38 min)

Business question: *Build a **customer 360** view — revenue from orders + engagement from clickstream.*

This is where Spark shines vs Pandas on large data: distributed hash joins.

In [ ]:
customer_revenue = (
    enriched
    .groupBy("customer_id")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.count("*").alias("orders"),
    )
)

customer_360 = (
    customer_revenue
    .join(actions_by_customer, on="customer_id", how="left")
    .orderBy(F.desc("total_revenue"))
)

print("Customer 360 — PySpark left join")
customer_360.show(8, truncate=False)

In [ ]:
# Pandas equivalent
customer_revenue_pd = (
    enriched_pd.groupby("customer_id")["revenue"]
    .agg(total_revenue="sum", orders="count")
    .round(2)
    .reset_index()
)
customer_360_pd = customer_revenue_pd.merge(actions_pd, on="customer_id", how="left")
customer_360_pd.sort_values("total_revenue", ascending=False).head(8)

In [ ]:
# Join orders with clickstream events on product_id (inner join — only products with both)
product_engagement = (
    orders.alias("o")
    .join(
        clickstream.filter(F.col("product_id").isNotNull()).alias("c"),
        F.col("o.product_id") == F.col("c.product_id"),
        "inner",
    )
    .select("o.product_name", "o.category", "c.action", "c.customer_id")
    .groupBy("product_name", "category", "action")
    .count()
    .orderBy(F.desc("count"))
)

print("Product-level engagement (inner join)")
product_engagement.show(truncate=False)

| Join type | Use when |
|-----------|----------|
| `inner` | Only matching keys |
| `left` | Keep all rows from left table |
| `broadcast(small_df)` | Hint: copy small table to every executor (fast for lookup tables) |

---
## Module 7 — Window functions (38–46 min)

Business questions:
1. Rank customers by revenue **within each city**
2. Running total revenue **over time**

In [ ]:
from pyspark.sql.window import Window

city_window = Window.partitionBy("city").orderBy(F.desc("revenue"))

ranked_in_city = (
    enriched
    .withColumn("city_rank", F.row_number().over(city_window))
    .select("city", "customer_id", "product_name", "revenue", "city_rank")
    .orderBy("city", "city_rank")
)

print("Top orders per city — window rank")
ranked_in_city.show(10, truncate=False)

In [ ]:
# Pandas equivalent — groupby + rank
enriched_pd["city_rank"] = (
    enriched_pd.groupby("city")["revenue"]
    .rank(method="first", ascending=False)
    .astype(int)
)
enriched_pd.sort_values(["city", "city_rank"]).head(10)

In [ ]:
date_window = Window.orderBy("order_date", "order_id")

running_total = (
    enriched
    .select("order_date", "order_id", "revenue")
    .withColumn("running_revenue", F.round(F.sum("revenue").over(date_window), 2))
)

print("Running total revenue by order date")
running_total.show(10, truncate=False)

In [ ]:
# Pandas equivalent
running_pd = enriched_pd.sort_values(["order_date", "order_id"]).copy()
running_pd["running_revenue"] = running_pd["revenue"].cumsum().round(2)
running_pd[["order_date", "order_id", "revenue", "running_revenue"]].head(10)

**Why windows matter:** Rankings, moving averages, sessionization, and funnel analysis all use window functions. Pandas `groupby().apply()` loops in Python; Spark runs windows in optimized JVM code.

---
## Module 8 — Spark SQL (46–52 min)

Analysts often prefer SQL. Register DataFrames as **temp views** and query with `spark.sql()`.

In [ ]:
enriched.createOrReplaceTempView("orders_enriched")
clickstream.createOrReplaceTempView("clickstream")

top_categories_sql = spark.sql("""
    SELECT
        category,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(*) AS order_lines
    FROM orders_enriched
    GROUP BY category
    ORDER BY total_revenue DESC
""")

print("Same aggregation — Spark SQL")
top_categories_sql.show()

In [ ]:
# Text analytics on reviews — word frequency (SQL + explode)
reviews.createOrReplaceTempView("reviews_raw")

top_words_sql = spark.sql("""
    SELECT word, COUNT(*) AS cnt
    FROM (
        SELECT explode(split(lower(value), '\\W+')) AS word
        FROM reviews_raw
    )
    WHERE word <> ''
    GROUP BY word
    ORDER BY cnt DESC
    LIMIT 10
""")

print("Top review words — Spark SQL")
top_words_sql.show(truncate=False)

In [ ]:
# Pandas text equivalent (small file only)
with open(data_path("product_reviews.txt")) as f:
    text = f.read().lower()

import re
words = [w for w in re.split(r"\W+", text) if w]
pd.Series(words).value_counts().head(10)

**Production tip:** On AWS, the same SQL runs on **Amazon EMR**, **Glue**, or **Databricks** — your PySpark skills transfer directly.

---
## Module 9 — UDFs, null handling, arrays (bonus depth)

Prefer built-in `F` functions (Catalyst optimizer can push them down). Use **UDFs** only when necessary.

In [ ]:
def revenue_tier(amount):
    if amount >= 100:
        return "high"
    if amount >= 50:
        return "medium"
    return "low"

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

tier_udf = udf(revenue_tier, StringType())

with_tier = enriched.withColumn("revenue_tier", tier_udf(F.col("revenue")))
with_tier.groupBy("revenue_tier").count().orderBy("revenue_tier").show()

In [ ]:
# Built-in alternative — faster, no Python serialization per row
with_tier_native = enriched.withColumn(
    "revenue_tier",
    F.when(F.col("revenue") >= 100, "high")
     .when(F.col("revenue") >= 50, "medium")
     .otherwise("low")
)
with_tier_native.groupBy("revenue_tier").count().orderBy("revenue_tier").show()

---
## Module 10 — Lazy evaluation, cache, explain (52–56 min)

In [ ]:
# enriched is reused in multiple downstream queries — cache it
enriched.cache()
print("Cached rows:", enriched.count())  # materializes cache

print("\nPhysical plan for customer 360 join:")
customer_360.explain(mode="formatted")

| Concept | What it means |
|---------|---------------|
| **Lazy transform** | `filter`, `join`, `groupBy` — plan only |
| **Action** | `show`, `count`, `write` — runs the job |
| **Cache** | Keep DataFrame in memory after first action |
| **explain()** | See optimized physical plan (shuffle, broadcast) |

Open **Spark UI → SQL / Jobs** at http://localhost:4040 to see stages and shuffle bytes.

---
## Module 11 — Write output & full pipeline (56–60 min)

Production pipelines **write once** to Parquet (columnar, compressed) on S3/HDFS.

In [ ]:
output_dir = spark_dir / "output" / "customer_360"
output_dir.mkdir(parents=True, exist_ok=True)

(
    customer_360
    .coalesce(1)  # single file for demo; production uses many partitions
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(output_dir))
)

written = list(output_dir.glob("part-*.csv"))
print(f"Wrote customer 360 report to {output_dir}")
print(f"Files: {[f.name for f in written]}")

# Read back — same as reading from S3 in production
spark.read.option("header", True).csv(str(output_dir)).show(5, truncate=False)

In [ ]:
# Cleanup
enriched.unpersist()
stop_spark(spark)
print("Spark session stopped.")

---
## Summary — what you practiced in 1 hour

| Capability | PySpark API | Pandas equivalent |
|------------|-------------|-------------------|
| Load CSV | `spark.read.csv()` | `pd.read_csv()` |
| Filter / select | `.filter()`, `.select()` | boolean indexing |
| New columns | `.withColumn()` | `df["col"] = ...` |
| Aggregate | `.groupBy().agg()` | `.groupby().agg()` |
| Pivot | `.groupBy().pivot()` | `.pivot_table()` |
| Join | `.join()` | `.merge()` |
| Window | `Window.partitionBy()` | `.groupby().rank()` / `.cumsum()` |
| SQL | `createOrReplaceTempView` + `spark.sql` | N/A (use DuckDB/SQL separately) |
| Performance | `.cache()`, `.explain()` | N/A at scale |
| Export | `.write.parquet()` / `.csv()` | `.to_csv()` |

### When to reach for PySpark

- Data **does not fit in RAM** on one machine
- Pipeline runs on a **schedule** (EMR, Glue, Databricks)
- You need **joins + windows + SQL** in one engine

### Next steps

| Resource | Path |
|----------|------|
| MapReduce → Spark pivot lab | [Spark-Pivot-Guide.ipynb](./Spark-Pivot-Guide.ipynb) |
| Written guide | [SPARK-STUDENT-GUIDE.md](./SPARK-STUDENT-GUIDE.md) |
| Hadoop cluster class | [../Hadoop-Local-Cluster-Class.ipynb](../Hadoop-Local-Cluster-Class.ipynb) |